In [1]:
!pip install torch-geometric
!pip install rdkit selfies tqdm

In [1]:
import torch
import pickle
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import selfies as sf
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, DataStructs, rdmolops, Descriptors, rdMolDescriptors, QED
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import math
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from sklearn.metrics import r2_score
from IPython.display import display
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import to_dense_batch, to_dense_adj
from torch_geometric.loader import DataLoader
import seaborn as sns
import os
import re
torch.cuda.empty_cache()

In [2]:
df = pd.read_csv('/content/data/raw/smiles_selfies_full.csv')

In [5]:
df.head()

,smiles,selfies
0,O=S(O)c1cc2c(cc1F)OC(c1ccc(F)cc1F)(c1ccc(F)cc1...,[O][=S][Branch1][C][O][C][=C][C][=C][Branch1][...
1,CN(C)Cc1cccc(C2Nc3cccc4c(=O)[nH]nc(c34)C2c2ccc...,[C][N][Branch1][C][C][C][C][=C][C][=C][C][Bran...
2,O=C(N[C@@H](CO)c1nc2cc(Cl)ccc2[nH]1)c1ccc(C(=O...,[O][=C][Branch2][Ring1][#Branch1][N][C@@H1][Br...
3,O=C(Cn1cc(I)cn1)N1CCCc2c1cnn2-c1ccc(F)cc1,[O][=C][Branch1][N][C][N][C][=C][Branch1][C][I...
4,Cc1ccc(-c2ccnc(Cl)c2)n1CC(=O)OCc1ccccc1,[C][C][=C][C][=C][Branch1][N][C][=C][C][=N][C]...


In [3]:
# convert to graphs
SUPPORTED_ATOMS = [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]

def atom_to_feature_vector(atom):
    """zamienia atom na wektor cech, ten wektor będzie potem nodem w grafie"""
    atomic_num = atom.GetAtomicNum()
    # one hot encoding
    return [1 if atomic_num == atom_type else 0 for atom_type in SUPPORTED_ATOMS]

def smiles_to_graph(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return None

    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append(atom_to_feature_vector(atom))
    x = torch.tensor(atom_features, dtype=torch.float)

    edges = []
    for bond in mol.GetBonds():
        edges.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
        edges.append((bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()))

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    data = Data(x=x, edge_index=edge_index)
    return data


In [7]:
smiles = "CC(CC)O"
graph = smiles_to_graph(smiles)
print(graph)
print(graph.x)
print(graph.edge_index)

Data(x=[5, 10], edge_index=[2, 8])
tensor([[0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]])
tensor([[0, 1, 1, 2, 2, 3, 1, 4],
        [1, 0, 2, 1, 3, 2, 4, 1]])


In [9]:
GRAPH_PATH = '/content/graphs.pt'

if os.path.exists(GRAPH_PATH):
    graph_list = torch.load(GRAPH_PATH)
else:
    graph_list = []
    for index, row in tqdm(df.iterrows(), total=len(df)):
        smiles = row['smiles']
        graph = smiles_to_graph(smiles)
        graph_list.append(graph)
    torch.save(graph_list, GRAPH_PATH)

  0%|          | 0/794403 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
torch.save(graph_list, '/content/graphs.pt')

In [4]:
# split dataset
graph_train, graph_temp = train_test_split(graph_list, test_size=0.2, random_state=42, shuffle=True)
graph_val, graph_test = train_test_split(graph_temp, test_size=0.5, random_state=42, shuffle=True)

class GraphDataset(Dataset):
    def __init__(self, graph_list):
        self.graphs = graph_list

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, index):
        return self.graphs[index]

train_dataset = GraphDataset(graph_train)
val_dataset = GraphDataset(graph_val)
test_dataset = GraphDataset(graph_test)

NameError: name 'graph_list' is not defined

In [5]:
# model
class GraphVAE(nn.Module):
    def __init__(self, latent_dim, max_nodes=37, in_channels=10):
        super().__init__()
        self.latent_dim = latent_dim
        self.max_nodes = max_nodes
        self.in_channels = in_channels

        # Encoder
        self.conv1 = GCNConv(in_channels, 64)
        self.conv2 = GCNConv(64, 128)

        # Latent space
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # Decoder
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
        )

        self.decoder_node = nn.Linear(256, max_nodes * in_channels)
        self.decoder_adj = nn.Linear(256, max_nodes * max_nodes)

    def encode(self, x, edge_index, batch):
        h = self.conv1(x, edge_index).relu()
        h = self.conv2(h, edge_index).relu()
        h_graph = global_mean_pool(h, batch)

        mu = self.fc_mu(h_graph)
        logvar = self.fc_logvar(h_graph)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu

    def decode(self, z):
        h = self.decoder_fc(z)

        recon_x = self.decoder_node(h).view(-1, self.max_nodes, self.in_channels)
        recon_adj = self.decoder_adj(h).view(-1, self.max_nodes, self.max_nodes)
        recon_adj = torch.sigmoid(recon_adj)

        return recon_x, recon_adj

    def forward(self, x, edge_index, batch):
        mu, logvar = self.encode(x, edge_index, batch)
        z = self.reparameterize(mu, logvar)
        recon_x, recon_adj = self.decode(z)

        # padding to max_nodes
        true_x, _ = to_dense_batch(x, batch, max_num_nodes=self.max_nodes)
        true_adj = to_dense_adj(edge_index, batch, max_num_nodes=self.max_nodes)

        return recon_x, recon_adj, true_x, true_adj, mu, logvar

def vae_graph_loss(recon_x, recon_adj, true_x, true_adj, mu, logvar, beta=0.01, edge_weight=1.0):
    in_channels = true_x.size(-1)
    logits = recon_x.reshape(-1, in_channels)
    targets = true_x.argmax(dim=-1).reshape(-1)

    # loss for atoms
    atom_loss = F.cross_entropy(logits, targets)

    # loss for edges
    edge_loss = F.binary_cross_entropy(recon_adj.reshape(-1), true_adj.reshape(-1))

    recon_loss = atom_loss + edge_weight * edge_loss

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    total_loss = recon_loss + beta * kl

    return total_loss, atom_loss.item(), edge_loss.item(), kl.item()



In [16]:
# training
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

latent_dim = 64
max_nodes = 37
in_channels = 10
model = GraphVAE(latent_dim, max_nodes=max_nodes, in_channels=in_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

beta = 0.01
edge_weight = 1.0
num_epochs = 10
history = {'train_loss': [], 'atom_loss': [], 'edge_loss': [], 'kl_loss': [], 'val_loss': []}
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

best_val_loss = float('inf')

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    total_atom_loss = 0
    total_edge_loss = 0
    total_kl_loss = 0
    pbar = tqdm(train_loader)
    for graph in pbar:
        x, edge_index, batch = graph.x.to(device), graph.edge_index.to(device), graph.batch.to(device)

        optimizer.zero_grad()

        recon_x, recon_adj, true_x, true_adj, mu, logvar = model(x, edge_index, batch)

        loss, atom_l, edge_l, kl_l = vae_graph_loss(
            recon_x, recon_adj, true_x, true_adj, mu, logvar, beta=beta, edge_weight=edge_weight
        )
        loss.backward()
        optimizer.step()

        pbar.set_postfix({
            "atom": f"{atom_l:.3f}",
            "edge": f"{edge_l:.3f}",
            "kl": f"{kl_l:.3f}",
            "tot": f"{loss.item():.3f}",
        })

        total_loss += loss.item()
        total_atom_loss += atom_l
        total_edge_loss += edge_l
        total_kl_loss += kl_l

    with torch.no_grad():
        model.eval()
        val_loss = 0
        for graph in val_loader:
            x, edge_index, batch = graph.x.to(device), graph.edge_index.to(device), graph.batch.to(device)
            recon_x, recon_adj, true_x, true_adj, mu, logvar = model(x, edge_index, batch)

            loss, _, _, _ = vae_graph_loss(
                recon_x, recon_adj, true_x, true_adj, mu, logvar, beta=beta, edge_weight=edge_weight
            )
            val_loss += loss.item()

    total_loss /= len(train_loader)
    total_atom_loss /= len(train_loader)
    total_edge_loss /= len(train_loader)
    total_kl_loss /= len(train_loader)
    val_loss /= len(val_loader)

    if val_loss < best_val_loss:
        MODEL_PATH = '/content/best_graph_vae.pt'
        torch.save(model.state_dict(), MODEL_PATH)
        best_val_loss = val_loss

    history['train_loss'].append(total_loss)
    history['atom_loss'].append(total_atom_loss)
    history['edge_loss'].append(total_edge_loss)
    history['kl_loss'].append(total_kl_loss)
    history['val_loss'].append(val_loss)
    print(f"Epoch {epoch}/{num_epochs} - Train Loss: {total_loss:.4f} (Atom: {total_atom_loss:.4f}, Edge: {total_edge_loss:.4f}, KL: {total_kl_loss:.4f}) - Val Loss: {val_loss:.4f}")

  0%|          | 0/19861 [00:00<?, ?it/s]

Epoch 1/10 - Train Loss: 0.7811 (Atom: 0.7048, Edge: 0.0647, KL: 1.1601) - Val Loss: 0.7213


  0%|          | 0/19861 [00:00<?, ?it/s]

Epoch 2/10 - Train Loss: 0.7058 (Atom: 0.6301, Edge: 0.0616, KL: 1.4050) - Val Loss: 0.6850


  0%|          | 0/19861 [00:00<?, ?it/s]

...

Epoch 8/10 - Train Loss: 0.6029 (Atom: 0.5296, Edge: 0.0568, KL: 1.6515) - Val Loss: 0.5952


  0%|          | 0/19861 [00:00<?, ?it/s]

Epoch 9/10 - Train Loss: 0.5966 (Atom: 0.5235, Edge: 0.0565, KL: 1.6566) - Val Loss: 0.6013


  0%|          | 0/19861 [00:00<?, ?it/s]

Epoch 10/10 - Train Loss: 0.5911 (Atom: 0.5181, Edge: 0.0563, KL: 1.6671) - Val Loss: 0.5969


In [6]:
!curl -o sascorer.py https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py

# Import deskryptorów i pobranego modułu
from rdkit.Chem import Descriptors
import sascorer

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  5913  100  5913    0     0  18889      0 --:--:-- --:--:-- --:--:-- 18951


In [7]:
# =====================================================================
# WCZYTYWANIE ZAPISANEGO MODELU GraphVAE
# =====================================================================

# 1. Definiujemy parametry strukturalne (muszą być identyczne jak przy treningu!)
latent_dim = 64
max_nodes = 37
in_channels = 10

# 2. Wybieramy urządzenie (GPU jeśli dostępne, inaczej CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Wczytuję model na urządzenie: {device}")

# 3. Tworzymy "pustą" instancję modelu i przenosimy ją na właściwe urządzenie
loaded_model = GraphVAE(latent_dim, max_nodes=max_nodes, in_channels=in_channels).to(device)

# 4. Ścieżka do zapisanego pliku z wagami
MODEL_PATH = '/content/best_graph_vae.pt'

# 5. Ładowanie wag z obsługą mapowania urządzeń (bezpieczne przenoszenie między GPU a CPU)
if os.path.exists(MODEL_PATH):
    # map_location dba o to, żeby model trenowany na GPU wczytał się nawet, jeśli teraz odpalisz go na CPU
    loaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

    # 6. KLUCZOWE: Przełączenie modelu w tryb ewaluacji (testowania)
    loaded_model.eval()
    print("--> Model wczytany pomyślnie i gotowy do generowania!")
else:
    print(f"[BŁĄD] Plik {MODEL_PATH} nie istnieje w tej ścieżce. Sprawdź panel boczny Colaba!")

Wczytuję model na urządzenie: cuda
--> Model wczytany pomyślnie i gotowy do generowania!


In [8]:
model = loaded_model

In [9]:
# ============================================================
# EVALUATION BLOCK — GraphVAE
# ============================================================

import math
import sascorer  # już pobrany wyżej
from collections import defaultdict
from rdkit.Chem import RWMol, Atom, BondType

# ── 1. helpers ──────────────────────────────────────────────

SUPPORTED_ATOMS = [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]  # musi zgadzać się z treningiem

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
]

def graph_to_mol(recon_x, recon_adj, threshold=0.5):
    """
    recon_x   : (max_nodes, in_channels) — logity / prawdopodobieństwa typów atomów
    recon_adj : (max_nodes, max_nodes)   — prawdopodobieństwa krawędzi (po sigmoid)
    Zwraca mol RDKit lub None jeśli niepoprawny.
    """
    atom_types = recon_x.argmax(dim=-1).cpu().numpy()   # indeks w SUPPORTED_ATOMS
    adj        = recon_adj.cpu().numpy()

    rwmol = RWMol()
    node_map = {}  # idx_w_macierzy → idx_w_mol

    # dodaj atomy (pomijaj atomy z indeksem 0 jeśli chcesz ignorować "padding")
    for i, atom_idx in enumerate(atom_types):
        atomic_num = SUPPORTED_ATOMS[atom_idx]
        if atomic_num == 0:          # padding / placeholder
            continue
        mol_idx = rwmol.AddAtom(Atom(int(atomic_num)))
        node_map[i] = mol_idx

    # dodaj krawędzie (górny trójkąt żeby nie dublować)
    added = set()
    for i in range(len(atom_types)):
        for j in range(i + 1, len(atom_types)):
            if i not in node_map or j not in node_map:
                continue
            if adj[i, j] >= threshold:
                pair = (node_map[i], node_map[j])
                if pair not in added:
                    rwmol.AddBond(pair[0], pair[1], Chem.rdchem.BondType.SINGLE)
                    added.add(pair)

    try:
        mol = rwmol.GetMol()
        Chem.SanitizeMol(mol)
        return mol
    except Exception:
        return None


def sample_mols_from_graphvae(model, n=1000, batch_size=256, threshold=0.5):
    """Losuje z prioru N(0,1), dekoduje i zamienia na mole."""
    model.eval()
    mols = []
    with torch.no_grad():
        for i in range(0, n, batch_size):
            bs = min(batch_size, n - i)
            z  = torch.randn(bs, model.latent_dim).to(device)
            recon_x, recon_adj = model.decode(z)          # (bs, max_nodes, in_channels / max_nodes, max_nodes)
            for k in range(bs):
                mol = graph_to_mol(recon_x[k], recon_adj[k], threshold=threshold)
                mols.append(mol)
    return mols


def get_molecule_properties(mol):
    return {
        "molWt":               Descriptors.MolWt(mol),
        "HeavyAtomCount":      mol.GetNumHeavyAtoms(),
        "cLogP":               Descriptors.MolLogP(mol),
        "TPSA":                Descriptors.TPSA(mol),
        "HBD":                 rdMolDescriptors.CalcNumHBD(mol),
        "HBA":                 rdMolDescriptors.CalcNumHBA(mol),
        "NumRotatableBonds":   rdMolDescriptors.CalcNumRotatableBonds(mol),
        "RingCount":           rdMolDescriptors.CalcNumRings(mol),
        "AromaticRingCount":   rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3":        rdMolDescriptors.CalcFractionCSP3(mol),
        "NumSpiroAtoms":       rdMolDescriptors.CalcNumSpiroAtoms(mol),
        "NumBridgeheadAtoms":  rdMolDescriptors.CalcNumBridgeheadAtoms(mol),
        "BertzCT":             Descriptors.BertzCT(mol),
        "QED":                 QED.qed(mol),
        "SA-score":            sascorer.calculateScore(mol),
    }


# ── 2. generowanie ──────────────────────────────────────────

N_SAMPLE = 2000          # ile cząsteczek chcesz wygenerować
ADJ_THRESHOLD = 0.5      # próg na sigmoid(adj) żeby uznać krawędź za istniejącą

print(f"Generowanie {N_SAMPLE} cząsteczek z prioru...")
sampled_mols = sample_mols_from_graphvae(model, n=N_SAMPLE, threshold=ADJ_THRESHOLD)

valid_mols = [m for m in sampled_mols if m is not None]
validity   = len(valid_mols) / len(sampled_mols) * 100
print(f"Validity: {len(valid_mols)}/{len(sampled_mols)} = {validity:.1f}%")


# ── 3. właściwości wygenerowanych cząsteczek ────────────────

Sampled_properties = []
for mol in valid_mols:
    try:
        props = get_molecule_properties(mol)
        Sampled_properties.append(props)
    except Exception:
        Sampled_properties.append(None)

prop_dict = defaultdict(list)
for props in Sampled_properties:
    if props is None:
        continue
    for k, v in props.items():
        if v is not None and not np.isnan(v):
            prop_dict[k].append(v)

print(f"Cząsteczki z policzonymi właściwościami: {len([p for p in Sampled_properties if p is not None])}")


# ── 4. wczytaj dataset referencyjny (dostosuj ścieżkę!) ─────

Y = pd.read_csv('/content/data/raw/smiles_selfies_full.csv')   # ← zmień jeśli masz osobny plik z właściwościami

# Jeśli Y to surowe SMILES — policz właściwości na fly:
if 'molWt' not in Y.columns:
    print("Liczę właściwości dla datasetu referencyjnego...")
    ref_props_list = []
    for smi in tqdm(Y['smiles'].dropna()):
        mol = Chem.MolFromSmiles(smi)
        if mol:
            try:
                ref_props_list.append(get_molecule_properties(mol))
            except Exception:
                ref_props_list.append(None)
        else:
            ref_props_list.append(None)
    Y_props = pd.DataFrame([p for p in ref_props_list if p is not None])
else:
    Y_props = Y  # już ma właściwości


# ── 5. wykresy — porównanie Dataset vs Generated ────────────

cols        = list(Y_props.columns)
n_cols_grid = 3
n_rows      = math.ceil(len(cols) / n_cols_grid)

fig, axes = plt.subplots(n_rows, n_cols_grid, figsize=(5 * n_cols_grid, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    gen_vals  = prop_dict.get(col, [])
    data_vals = Y_props[col].dropna().values

    if len(gen_vals) == 0 or len(data_vals) == 0:
        axes[i].set_visible(False)
        continue

    combined = np.concatenate([data_vals, gen_vals])
    bins     = np.linspace(combined.min(), combined.max(), 50)

    axes[i].hist(data_vals, bins=bins, alpha=0.5, label="Dataset",   density=True, color="steelblue")
    axes[i].hist(gen_vals,  bins=bins, alpha=0.5, label="Generated", density=True, color="tomato")
    axes[i].set_title(col)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Density")
    axes[i].grid(alpha=0.3)
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("GraphVAE — właściwości wygenerowanych vs referencyjnych cząsteczek", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


# ── 6. summary tabela ───────────────────────────────────────

summary_rows = []
for col in cols:
    gen_vals  = prop_dict.get(col, [])
    data_vals = Y_props[col].dropna().values
    if len(gen_vals) == 0:
        continue
    summary_rows.append({
        "Property":        col,
        "Dataset mean":    np.mean(data_vals),
        "Dataset std":     np.std(data_vals),
        "Generated mean":  np.mean(gen_vals),
        "Generated std":   np.std(gen_vals),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Property").round(3)
display(summary_df)


# ── 7. validity / uniqueness / novelty ──────────────────────

# Uniqueness
gen_smiles  = [Chem.MolToSmiles(m) for m in valid_mols]
unique_smi  = set(gen_smiles)
uniqueness  = len(unique_smi) / len(gen_smiles) * 100 if gen_smiles else 0

# Novelty (względem datasetu treningowego)
train_smiles_set = set(df['smiles'].dropna().tolist())
novel_smi   = [s for s in unique_smi if s not in train_smiles_set]
novelty     = len(novel_smi) / len(unique_smi) * 100 if unique_smi else 0

print("\n── Metryki generatywne ──")
print(f"  Validity:    {validity:.1f}%")
print(f"  Uniqueness:  {uniqueness:.1f}%  ({len(unique_smi)}/{len(gen_smiles)})")
print(f"  Novelty:     {novelty:.1f}%  ({len(novel_smi)}/{len(unique_smi)})")

Generowanie 2000 cząsteczek z prioru...


[22:23:16] Explicit valence for atom # 0 S, 13, is greater than permitted
[22:23:16] Explicit valence for atom # 1 N, 4, is greater than permitted
[22:23:16] Explicit valence for atom # 10 H, 2, is greater than permitted
[22:23:16] Explicit valence for atom # 4 Br, 2, is greater than permitted
[22:23:16] Explicit valence for atom # 1 C, 5, is greater than permitted
[22:23:16] Explicit valence for atom # 1 C, 5, is greater than permitted
[22:23:16] Explicit valence for atom # 8 H, 2, is greater than permitted
[22:23:16] Explicit valence for atom # 2 C, 6, is greater than permitted
[22:23:16] Explicit valence for atom # 12 H, 2, is greater than permitted
[22:23:16] Explicit valence for atom # 0 Cl, 3, is greater than permitted
[22:23:16] Explicit valence for atom # 4 Br, 2, is greater than permitted
[22:23:16] Explicit valence for atom # 2 Cl, 2, is greater than permitted
[22:23:16] Explicit valence for atom # 7 H, 2, is greater than permitted
[22:23:16] Explicit valence for atom # 1 C, 

Validity: 1865/2000 = 93.2%


Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not removing hydrogen atom without neighbors
[22:23:24] WARNING: not r

Cząsteczki z policzonymi właściwościami: 0
Liczę właściwości dla datasetu referencyjnego...


  0%|          | 0/794403 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [19]:
with torch.no_grad():
    model.eval()
    # Generujemy 100 próbnych wektorów
    z_test = torch.randn((100, latent_dim)).to(device)

    for prog in [0.3, 0.5, 0.7, 0.8, 0.9]:
        test_mols = latents_to_mol(model, z_test, cutoff_edge=prog)
        sukcesy = sum(1 for m in test_mols if m is not None)
        print(f"Dla progu cutoff_edge = {prog:.1f} pomyślnie przeszło: {sukcesy}/100 cząsteczek")

Dla progu cutoff_edge = 0.3 pomyślnie przeszło: 88/100 cząsteczek
Dla progu cutoff_edge = 0.5 pomyślnie przeszło: 95/100 cząsteczek
Dla progu cutoff_edge = 0.7 pomyślnie przeszło: 96/100 cząsteczek
Dla progu cutoff_edge = 0.8 pomyślnie przeszło: 96/100 cząsteczek
Dla progu cutoff_edge = 0.9 pomyślnie przeszło: 98/100 cząsteczek


[21:57:59] Explicit valence for atom # 3 O, 3, is greater than permitted
[21:57:59] Explicit valence for atom # 2 F, 2, is greater than permitted
[21:57:59] Explicit valence for atom # 5 C, 5, is greater than permitted
[21:57:59] Explicit valence for atom # 1 O, 3, is greater than permitted
[21:57:59] Explicit valence for atom # 12 C, 6, is greater than permitted
[21:57:59] Explicit valence for atom # 1 C, 5, is greater than permitted
[21:57:59] Explicit valence for atom # 13 F, 2, is greater than permitted
[21:57:59] Explicit valence for atom # 2 F, 2, is greater than permitted
[21:57:59] Explicit valence for atom # 1 N, 4, is greater than permitted
[21:57:59] Explicit valence for atom # 3 F, 2, is greater than permitted
[21:57:59] Explicit valence for atom # 1 C, 5, is greater than permitted
[21:57:59] Explicit valence for atom # 2 Cl, 3, is greater than permitted
[21:57:59] Explicit valence for atom # 2 F, 2, is greater than permitted
[21:57:59] Explicit valence for atom # 1 C, 5, i